# Streaming Live League games with Redis

In [4]:
import redis
from src.fetch_data.fetch_live_leagues import retrieve_live_league_games
import time 

In [3]:
# Set up redis server
r = redis.Redis(
    host='localhost',
    port=6379,
    decode_responses=True
)

r.ping()

True

In [ ]:
# Constants
match_set = 'live_match_ids'

In [ ]:
import redis.exceptions


def poll_live_matches():
    res = retrieve_live_league_games()
    if not res:
        return 0
    curr_match_ids = [item['match_id'] for item in res]
    curr_match_details = {item['match_id']: item for item in res}
    tmp_key = f'{match_set}:temp'
    
    
    # Delete matches from last poll and update with curr poll results
    r.delete(tmp_key)
    
    if curr_match_ids:
        # Get new matches ids and update match_set to curr polling
        r.sadd(tmp_key, *curr_match_ids)
        new_matches = r.sdiff(tmp_key, match_set)
        r.rename(tmp_key, match_set)
        
        # Add match details for new matches, and add to event stream
        pipe = r.pipeline()
        for match_id, match_details in curr_match_details.items():
            if match_id in new_matches:
                pipe.hset(f'match_details:{match_id}', mapping={**match_details,'status':'ongoing'})
                pipe.xadd('ongoing_matches', {'match_id': match_id, 'timestamp':time.strftime('%Y-%m-%d %H:%M:%S')})
        
        pipe.execute()
        print(f'{len(new_matches)} new matches have been added')
    
    
    # Add consumer group for Ongoing matches
    try:
        r.xgroup_create('ongoing_matches', 'prediction_group', id='0', mkstream=True)
    except redis.exceptions.ResponseError as e:
        if 'BUSYGROUP' in str(e):
            print("prediction_group already exists")
        else:
            raise e
    
    events = r.xreadgroup('prediction_group', 'consumer1', {'ongoing_matches': '>'})
    
    pipe = r.pipeline()
    
    for stream_name, stream_events in events:
        for event_id, data in stream_events:
            match_id = data['match_id']
            match_details = curr_match_details.get(match_id)
            game_duration = match_details.get('game_duration', 0)
            
            if game_duration > 0:
                print(f"Feature Engineered for match {match_id}")
                print(f"Predictions Made for {match_id}")
                pipe.hset(f'match_details:{match_id}', 'status', 'predicted')
                pipe.xadd('predicted_matches', 
                          {'match_id':match_id, 'timestamp':time.strftime('%Y-%m-%d %H:%M:%S')})
                pipe.xack('ongoing_matches','prediction_group', event_id)
                
    pipe.execute()
    
       # Add consumer group for predicted matches
    try:
        r.xgroup_create('predicted_matches', 'completion_group', id='0', mkstream=True)
    except redis.exceptions.ResponseError as e:
        if 'BUSYGROUP' in str(e):
            print("completion group already exists")
        else:
            raise e
    
    events = r.xreadgroup('completion_group', 'consumer1', {'predicted_matches': '>'})
    
    pipe = r.pipeline()
    for stream_name, stream_events in events:
        for event_id, data in stream_events:
            match_id = data['match_id']
            if match_id not in curr_match_ids:
                outcome = get_match_outcome(match_id) # Function to simulate getting match outcome. 
                if outcome:
                    print(f"fetched match outcome for match {match_id}")
                    print(f"Stored updated and completed match details in database")
                    pipe.delete(f'match_details:{match_id}')
                    pipe.xack('predicted_matches', 'completion_group', event_id)
                
    pipe.execute()      



def get_match_outcome(match_id):
    return True  
            

